# HK Free Press News RAG Ingestion with LlamaIndex and ChromaDB

This notebook builds the ingestion side of your news chatbot. It reads text files from `data/hk_free_press_news`, cleans and normalizes each article, splits the articles into retrieval-friendly chunks, embeds those chunks with a local Hugging Face model, and stores them in a persistent ChromaDB collection named `news_chat`.

The flow is:

1. Install and import the required libraries.
2. Point the notebook at your text news folder and ChromaDB folder.
3. Load `.txt` news articles as LlamaIndex documents with useful metadata.
4. Chunk each article into retrieval-friendly nodes.
5. Insert the nodes into ChromaDB.
6. Run a small retrieval test to confirm the vector database is working.

In [ ]:
# Install the RAG ingestion dependencies in the active notebook kernel.
# Run this cell once, then restart the kernel if imports still fail.
%pip install -U llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface chromadb sentence-transformers tqdm psutil arize-phoenix arize-phoenix-otel openinference-instrumentation-llama-index

In [1]:
import re
import sys
import time
import unicodedata
from pathlib import Path

import chromadb
from chromadb.errors import NotFoundError
from llama_index.core import Settings, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document, TextNode
from llama_index.vector_stores.chroma import ChromaVectorStore
from tqdm.auto import tqdm

project_root = Path.cwd().resolve()
if not (project_root / "app.py").exists() and (project_root.parent / "app.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag import (
    DEFAULT_INGESTION_OBSERVABILITY_CONFIG,
    ChromaVectorStoreAdmin,
    IngestionPhoenixObserver,
    build_ingestion_trace_attributes,
    create_embedding_model,
    default_ingestion_run_config,
    setup_phoenix_observability,
)

resource module not available on Windows


## 1. Configure Paths and Embeddings

The news folder is your source corpus. Each `.txt` file is treated as one news article before chunking.

The ChromaDB folder is where the vector database is saved. The collection name now uses a multilingual E5 suffix so bilingual vectors stay separate from the older English-only collection.

This notebook uses `intfloat/multilingual-e5-base` because it supports English and Traditional Chinese with lower memory usage than larger multilingual embedding models.

In [2]:
INGESTION_RUN_CONFIG = default_ingestion_run_config(project_root)
INGESTION_OBSERVABILITY_CONFIG = DEFAULT_INGESTION_OBSERVABILITY_CONFIG

# Insertion-specific defaults live in src/rag/ingestion_config.py.
# The notebook reads them here so this cell stays short and easy to override for experiments.
PROJECT_ROOT = INGESTION_RUN_CONFIG.project_root
paths = INGESTION_RUN_CONFIG.paths
NEWS_DIRS = paths.news_dirs
CHROMA_DIR = paths.chroma_dir
COLLECTION_NAME = INGESTION_RUN_CONFIG.collection_name
CHUNK_SIZE = INGESTION_RUN_CONFIG.chunk_size
CHUNK_OVERLAP = INGESTION_RUN_CONFIG.chunk_overlap
EMBED_MODEL_NAME = INGESTION_RUN_CONFIG.embed_model_name

PHOENIX_ENABLED = INGESTION_OBSERVABILITY_CONFIG.phoenix_enabled
PHOENIX_ENDPOINT = INGESTION_OBSERVABILITY_CONFIG.phoenix_endpoint
PHOENIX_PROJECT_NAME = INGESTION_OBSERVABILITY_CONFIG.phoenix_project_name
PHOENIX_DATASET_NAME = INGESTION_OBSERVABILITY_CONFIG.phoenix_dataset_name
PHOENIX_PROJECT_DESCRIPTION = INGESTION_OBSERVABILITY_CONFIG.phoenix_project_description
PHOENIX_CHUNKING_METHOD = INGESTION_OBSERVABILITY_CONFIG.chunking_method

missing_news_dirs = [news_dir for news_dir in NEWS_DIRS if not news_dir.exists()]
if missing_news_dirs:
    missing_list = ", ".join(str(path) for path in missing_news_dirs)
    raise FileNotFoundError(f"News text folder(s) not found: {missing_list}")

# LlamaIndex uses Settings as global defaults for later index and query operations.
# create_embedding_model centralizes E5 query/passage prefixes so insertion and retrieval match.
Settings.embed_model = create_embedding_model(EMBED_MODEL_NAME)
Settings.llm = None

# Phoenix receives custom ingestion spans plus LlamaIndex spans created later in the notebook.
phoenix_status = setup_phoenix_observability(
    project_name=PHOENIX_PROJECT_NAME,
    endpoint=PHOENIX_ENDPOINT,
    enabled=PHOENIX_ENABLED,
    launch_server=False,
    auto_instrument=False,
    batch=True,
    raise_on_missing=False,
)
ingestion_observer = IngestionPhoenixObserver(
    collection_name=COLLECTION_NAME,
    embedding_model_name=EMBED_MODEL_NAME,
    phoenix_base_url=PHOENIX_ENDPOINT,
    dataset_name=PHOENIX_DATASET_NAME,
    enabled=phoenix_status.enabled,
)


def upsert_phoenix_project_description(base_url: str, project_name: str, description: str) -> None:
    """Create or update Phoenix project description so project context is visible in UI/API."""
    try:
        from phoenix.client import Client
    except (ImportError, ModuleNotFoundError):
        print("Phoenix client package not available; skip project description sync.")
        return

    client = Client(base_url=base_url)
    try:
        # Update description when project already exists.
        client.projects.update(project_name=project_name, description=description)
        print(f"Phoenix project description updated: {project_name}")
    except Exception:
        # Create project with description when it does not exist yet.
        client.projects.create(name=project_name, description=description)
        print(f"Phoenix project created with description: {project_name}")


if phoenix_status.enabled:
    upsert_phoenix_project_description(
        base_url=PHOENIX_ENDPOINT,
        project_name=PHOENIX_PROJECT_NAME,
        description=PHOENIX_PROJECT_DESCRIPTION,
    )

# Put ingestion configuration into a parent span so it is easy to read in Phoenix trace attributes.
INGESTION_TRACE_ATTRIBUTES = build_ingestion_trace_attributes(
    config=INGESTION_OBSERVABILITY_CONFIG,
    run_config=INGESTION_RUN_CONFIG,
)

# Daily ChromaDB operations helper: list collections, inspect metadata/samples, and delete collections.
vector_admin = ChromaVectorStoreAdmin(CHROMA_DIR)

print("News source folders:")
for news_dir in NEWS_DIRS:
    print(f"- {news_dir}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Collection name: {COLLECTION_NAME}")
print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"Phoenix status: {phoenix_status.message}")
print(f"Phoenix project: {PHOENIX_PROJECT_NAME}")
print(f"Phoenix dataset name: {PHOENIX_DATASET_NAME}")
print(f"Existing Chroma collections: {vector_admin.list_collection_names()}")

LLM is explicitly disabled. Using MockLLM.


c:\Users\jason\miniconda3\envs\ai\Lib\site-packages\authlib\_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


OpenTelemetry Tracing Details
|  Phoenix Project: rag-news-ingestion
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix project created with description: rag-news-ingestion
News source folders:
- C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news
- C:\Program Files\Studying\coding\RAG_project\data\hk01_news
- C:\Program Files\Studying\coding\RAG_project\data\the_standard_news
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Collection name: news_chat_multilingual_e5_base
Embedding model: intfloat/multilingual-e5-base
Phoenix status: Phoenix tracing is enabled. Run RAG queries, then op

In [3]:
# Daily vector database operations.
# Run this cell whenever you want to inspect ChromaDB before/after ingestion.
print("Collection count:", vector_admin.count_collections())
print("Collection names:", vector_admin.list_collection_names())

if COLLECTION_NAME in vector_admin.list_collection_names():
    summary = vector_admin.describe_collection(COLLECTION_NAME)
    print("Active collection summary:", summary)
    sample = vector_admin.sample_records(COLLECTION_NAME, limit=2)
    print("Sample ids:", sample.get("ids", []))
    print("Sample metadata:", sample.get("metadatas", []))
else:
    print(f"Collection {COLLECTION_NAME!r} does not exist yet. Run the insertion cell to create it.")

# To delete a collection intentionally, uncomment the next line.
# vector_admin.delete_collection("collection_name_to_delete", missing_ok=True)

Collection count: 4
Collection names: ['mini_wikipedia', 'news_chat', 'news_chat_multilingual_e5_base', 'test_collection_name']
Active collection summary: ChromaCollectionSummary(name='news_chat_multilingual_e5_base', count=256, metadata={'chunk_overlap': 120, 'embedding_model': 'intfloat/multilingual-e5-base', 'chunk_size': 800, 'source_folder_count': 3, 'source_folders': 'C:\\Program Files\\Studying\\coding\\RAG_project\\data\\hk01_news | C:\\Program Files\\Studying\\coding\\RAG_project\\data\\hk_free_press_news | C:\\Program Files\\Studying\\coding\\RAG_project\\data\\the_standard_news'})
Sample ids: ['8e30fe89-b65e-4356-8ef1-06e84364f16e', '54bc8f15-2161-49dd-a277-31f2bcd3fc3e']
Sample metadata: [{'ref_doc_id': '83d41e4c-cb52-4865-8392-08bfc42653fa', 'source_folder': 'hk01_news', 'file_path': 'C:\\Program Files\\Studying\\coding\\RAG_project\\data\\hk01_news\\01-06-2026_回憶的味道音樂劇2026香港｜門票攻略+購票連結+座位表.txt', 'chunk_overlap': 120, 'doc_id': '83d41e4c-cb52-4865-8392-08bfc42653fa', '_node

## 2. Load Text News Articles

Your new dataset is already stored as text files, so we do not need PDF extraction anymore. That is a good improvement for RAG: plain text is usually cleaner, faster to load, and less likely to produce unreadable characters than PDF parsing.

In this section, each `.txt` file becomes one LlamaIndex `Document`. The loader also extracts useful metadata from the file name:

- `article_date`: the date prefix at the start of the file name, such as `01-01-2026`
- `article_title`: the article title after the underscore
- `file_name` and `file_path`: the original source file information

This metadata is stored alongside each chunk in ChromaDB, so later retrieval can show where an answer came from.

In [4]:
from collections.abc import Iterable


def list_text_files(news_dirs: Iterable[Path]) -> list[Path]:
    """Return all text news files under all configured source folders."""
    # Use rglob so nested folders are included automatically if you organize files later.
    text_files: list[Path] = []
    for news_dir in news_dirs:
        text_files.extend(sorted(news_dir.rglob("*.txt")))
    return sorted(text_files)


def normalize_news_text(text: str) -> str:
    """Clean text-file artifacts while preserving the article content."""
    # Unicode normalization standardizes curly quotes, full-width characters, and ligatures.
    cleaned_text = unicodedata.normalize("NFKC", text or "")

    # Remove invisible or problematic characters that can hurt chunking and retrieval.
    cleaned_text = cleaned_text.replace("\x00", " ").replace("\ufeff", "").replace("\u00ad", "")

    # Normalize line endings first, then collapse excessive spaces and blank lines.
    cleaned_text = re.sub(r"\r\n?", "\n", cleaned_text)
    cleaned_text = re.sub(r"[ \t]+", " ", cleaned_text)
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)

    return cleaned_text.strip()


def parse_news_file_metadata(file_path: Path) -> dict[str, str]:
    """Extract article metadata from a news text file path."""
    # File names look like: 01-01-2026_Article title.txt
    # Keeping date, title, and source folder as metadata improves citation quality at retrieval time.
    file_stem = file_path.stem
    date_part, separator, title_part = file_stem.partition("_")

    article_date = date_part if separator else "unknown_date"
    article_title = title_part if separator else file_stem
    source_folder = file_path.parent.name

    return {
        "source_type": "news_txt",
        "source_folder": source_folder,
        "article_date": article_date,
        "article_title": article_title,
        "file_name": file_path.name,
        "file_path": str(file_path),
    }


def read_text_file(file_path: Path) -> str:
    """Read one news text file with a small encoding fallback."""
    # Most modern scraped text is UTF-8, but utf-8-sig also removes a BOM if present.
    for encoding in ("utf-8-sig", "utf-8", "cp1252"):
        try:
            return file_path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            continue

    # If all strict decoders fail, replace broken characters instead of stopping ingestion.
    return file_path.read_text(encoding="utf-8", errors="replace")


def load_news_documents(news_dirs: Iterable[Path], show_progress: bool = True) -> list[Document]:
    """Load text news articles into LlamaIndex Documents with cleaned text and metadata."""
    documents: list[Document] = []
    skipped_files: list[str] = []
    text_paths = list_text_files(news_dirs)

    file_iterator = text_paths
    if show_progress:
        file_iterator = tqdm(text_paths, desc="Loading and cleaning articles", unit="file")

    for text_path in file_iterator:
        raw_text = read_text_file(text_path)
        cleaned_text = normalize_news_text(raw_text)

        # Empty files should not become chunks because they cannot help retrieval.
        if not cleaned_text:
            skipped_files.append(text_path.name)
            continue

        # The Document is the article-level object that LlamaIndex will later split into chunks.
        documents.append(
            Document(
                text=cleaned_text,
                metadata=parse_news_file_metadata(text_path),
            )
        )

    if skipped_files:
        print("Warning: some text files were empty after cleaning and were skipped.")
        for file_name in skipped_files[:10]:
            print(f"- {file_name}")
        if len(skipped_files) > 10:
            print(f"... and {len(skipped_files) - 10} more skipped file(s)")

    return documents


def preview_news_documents(documents: list[Document], preview_count: int = 3) -> None:
    """Print a few loaded articles so you can quickly judge text quality."""
    for index, document in enumerate(documents[:preview_count], start=1):
        metadata = document.metadata
        print(f"\n--- Preview {index} ---")
        print(f"Source folder: {metadata.get('source_folder', 'unknown')}")
        print(f"Date: {metadata['article_date']}")
        print(f"Title: {metadata['article_title']}")
        print(document.text[:1200])


text_files = list_text_files(NEWS_DIRS)
print(f"Found {len(text_files)} text news file(s) across {len(NEWS_DIRS)} source folder(s).")
for text_file in text_files[:10]:
    print(f"- {text_file.name}")

if len(text_files) > 10:
    print(f"... and {len(text_files) - 10} more")


# Trace loading latency because it is runtime behavior; store per-document quality metrics as dataset records.
documents, document_load_latency = ingestion_observer.measure(
    "ingestion.load_documents",
    lambda: load_news_documents(NEWS_DIRS, show_progress=True),
    {"files.discovered": len(text_files)},
)
document_metrics = ingestion_observer.build_document_metrics(documents)
print(f"Loaded {len(documents)} news article document(s) in {document_load_latency:.2f}s.")
print(f"Prepared {len(document_metrics)} document metric record(s) for Phoenix dataset upload.")
preview_news_documents(documents)

Found 5254 text news file(s) across 3 source folder(s).
- 01-06-2026_回憶的味道音樂劇2026香港｜門票攻略+購票連結+座位表.txt
- 01-06-2026_張敬軒《軒動心靈》演唱會2026澳門站｜7月澳門美高梅劇院舉行.txt
- 01-06-2026_楊千嬅演唱會2026廣州｜門票攻略＋購票連結＋座位表.txt
- 02-04-2026_Kodaline演唱會2026香港加場｜門票攻略＋購票連結＋座位表.txt
- 02-04-2026_金俊秀演唱會2026香港丨門票優先公售攻略＋購票連結＋座位表.txt
- 02-06-2026_K歌之王陳輝陽作品演唱會2026澳門加場｜門票攻略+連結+座位表.txt
- 02-06-2026_STAYC見面會2026澳門｜門票攻略＋購票連結＋座位表.txt
- 02-06-2026_田井中彩智演唱會2026香港｜門票攻略＋購票連結＋座位表.txt
- 03-05-2026_&TEAM演唱會2026香港｜門票優先／公售攻略＋購票連結＋座位表.txt
- 03-06-2026_FTISLAND演唱會2026香港｜門票攻略＋購票連結＋座位表.txt
... and 5244 more


Loading and cleaning articles:   0%|          | 0/5254 [00:00<?, ?file/s]

- 17-01-2026_HKFP Monitor Jan 17, 2026 Jimmy Lai trial ignites courtroom queue rivalry.txt
- 31-01-2026_HKFP Monitor Jan 31, 2026 Democrats outside bars; lawmaker in hot water.txt
Loaded 5252 news article document(s) in 33.30s.
Prepared 5252 document metric record(s) for Phoenix dataset upload.

--- Preview 1 ---
Source folder: hk01_news
Date: 01-06-2026
Title: 回憶的味道音樂劇2026香港｜門票攻略+購票連結+座位表
《回憶的味道》音樂劇將於2026年8月6—16日在香港西灣河文娛中心劇院舉行,即刻看內文了解如何搶飛、場地座位表及交通!

>> 更多香港及澳門演唱會及活動介紹 <<

>> 立即購買!香港不同演唱會門票 <<

《回憶的味道》音樂劇2026香港 | 詳情

音樂劇日期:2026年8月6—16日 音樂劇時間:2026年8月6日至9日、8月11日至16日,晚上8時 2026年8月8日至9日、8月15日至16日,下午3時 音樂劇地點:西灣河文娛中心劇院 門票價錢:$880, $680 *全日制學生、60歲或以上高齡人士、綜合社會保障援助受惠人,以及殘疾人士及看護人可購買半價優惠門票。優惠票名額有限,售完即止。

《回憶的味道》音樂劇2026香港 | Mastercard優先購票

優先訂票平台:POPTICKET 門票發售時間:6月10日上午10時 限購:10張門票

《回憶的味道》音樂劇2026香港 | 門票公開發售

公開發售平台 :暫未公佈 公開開賣日期:暫未公佈 公開開賣時間:暫未公佈

--- Preview 2 ---
Source folder: hk01_news
Date: 01-06-2026
Title: 張敬軒《軒動心靈》演唱會2026澳門站｜7月澳門美高梅劇院舉行
張敬軒《軒動心靈》澳門站演唱會將於2026年7月4日(六)在美高梅劇院舉行,以私人Private Show 舉

## 3. Chunk the News Text

RAG works better when articles are split into chunks that are large enough to contain meaning but small enough to retrieve precisely.

Here we use sentence-aware chunking with overlap. The overlap is important for news articles because names, places, and quoted context may span chunk boundaries. Each chunk keeps the article metadata from the original text file, so retrieved chunks can still show the article date, title, and file name.

In [5]:
def build_text_nodes(
    documents: list[Document],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
    show_progress: bool = True,
) -> list[TextNode]:
    """Split loaded news documents into chunks that can be embedded and retrieved."""
    # SentenceSplitter tries to preserve sentence boundaries instead of cutting text blindly.
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    nodes = splitter.get_nodes_from_documents(documents)

    node_iterator = enumerate(nodes, start=1)
    if show_progress:
        node_iterator = enumerate(
            tqdm(nodes, desc="Annotating chunk metadata", unit="chunk"),
            start=1,
        )

    # Store chunk-level metadata to make retrieved news results easier to inspect later.
    for chunk_number, node in node_iterator:
        node.metadata["chunk_number"] = chunk_number
        node.metadata["chunk_size"] = chunk_size
        node.metadata["chunk_overlap"] = chunk_overlap

    return nodes


# Chunking latency is a span metric; chunk counts per document are also written into dataset records.
nodes, chunking_latency = ingestion_observer.measure(
    "ingestion.chunk_documents",
    lambda: build_text_nodes(documents, show_progress=True),
    {
        "documents.count": len(documents),
        "chunk.size": CHUNK_SIZE,
        "chunk.overlap": CHUNK_OVERLAP,
    },
)
ingestion_observer.record_chunking(documents, nodes, chunking_latency)

print(f"Loaded {len(documents)} news article document(s).")
print(f"Created {len(nodes)} text chunk(s) in {chunking_latency:.2f}s.")
print("\nPreview of first chunk:")
print(nodes[0].get_content()[:800] if nodes else "No chunks created.")

Annotating chunk metadata:   0%|          | 0/10972 [00:00<?, ?chunk/s]

Loaded 5252 news article document(s).
Created 10972 text chunk(s) in 9.88s.

Preview of first chunk:
《回憶的味道》音樂劇將於2026年8月6—16日在香港西灣河文娛中心劇院舉行,即刻看內文了解如何搶飛、場地座位表及交通!

>> 更多香港及澳門演唱會及活動介紹 <<

>> 立即購買!香港不同演唱會門票 <<

《回憶的味道》音樂劇2026香港 | 詳情

音樂劇日期:2026年8月6—16日 音樂劇時間:2026年8月6日至9日、8月11日至16日,晚上8時 2026年8月8日至9日、8月15日至16日,下午3時 音樂劇地點:西灣河文娛中心劇院 門票價錢:$880, $680 *全日制學生、60歲或以上高齡人士、綜合社會保障援助受惠人,以及殘疾人士及看護人可購買半價優惠門票。優惠票名額有限,售完即止。

《回憶的味道》音樂劇2026香港 | Mastercard優先購票

優先訂票平台:POPTICKET 門票發售時間:6月10日上午10時 限購:10張門票

《回憶的味道》音樂劇2026香港 | 門票公開發售

公開發售平台 :暫未公佈 公開開賣日期:暫未公佈 公開開賣時間:暫未公佈


In [6]:
RESET_COLLECTION = True


def get_chroma_collection(
    chroma_dir: Path,
    collection_name: str,
    reset_collection: bool = False,
    metadata: dict | None = None,
):
    """Create or load a persistent ChromaDB collection."""
    # PersistentClient writes vectors and metadata to disk instead of keeping them in memory.
    chroma_dir.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(chroma_dir))

    # Some ChromaDB versions raise NotFoundError when the collection is missing,
    # while older versions may raise ValueError. Catch both so reset stays safe.
    if reset_collection:
        try:
            client.delete_collection(collection_name)
            print(f"Deleted existing collection: {collection_name}")
        except (NotFoundError, ValueError):
            print(f"Collection did not exist yet: {collection_name}")

    # Chroma collection metadata must be scalar values (string/number/bool), not arrays.
    return client.get_or_create_collection(collection_name, metadata=metadata)


def sanitize_chroma_metadata(metadata: dict) -> dict:
    """Keep only Chroma-compatible scalar metadata values."""
    # Chroma rejects lists, dicts, and None values, so convert/drop them before insertion.
    scalar_metadata = {}
    for key, value in metadata.items():
        if isinstance(value, (str, int, float, bool)):
            scalar_metadata[key] = value
        elif value is not None:
            scalar_metadata[key] = str(value)
    return scalar_metadata


def build_or_update_vector_index(
    text_nodes: list[TextNode],
    chroma_dir: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
    reset_collection: bool = RESET_COLLECTION,
    batch_size: int = 256,
    show_progress: bool = True,
) -> VectorStoreIndex:
    """Embed nodes, write them to ChromaDB, and emit Phoenix ingestion metrics."""
    source_folders = sorted(str(news_dir) for news_dir in NEWS_DIRS)
    collection_metadata = {
        "embedding_model": EMBED_MODEL_NAME,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "source_folder_count": len(source_folders),
        # Serialize folders into one string because list metadata crashes Chroma.
        "source_folders": " | ".join(source_folders),
    }

    # ChromaVectorStore adapts the Chroma collection to LlamaIndex's vector store interface.
    collection = get_chroma_collection(chroma_dir, collection_name, reset_collection, metadata=collection_metadata)
    vector_store = ChromaVectorStore(chroma_collection=collection)

    if not text_nodes:
        return VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=Settings.embed_model)

    batch_starts = list(range(0, len(text_nodes), batch_size))
    batch_iterator = batch_starts
    if show_progress:
        batch_iterator = tqdm(batch_starts, desc="Embedding and writing Chroma batches", unit="batch")

    for batch_index, start in enumerate(batch_iterator, start=1):
        batch_nodes = text_nodes[start : start + batch_size]
        texts = [node.get_content() for node in batch_nodes]
        ids = [node.node_id for node in batch_nodes]
        metadatas = [sanitize_chroma_metadata(node.metadata or {}) for node in batch_nodes]

        # Embedding latency is traced separately from vector DB latency so Phoenix can show the bottleneck.
        embedding_start = time.perf_counter()
        embeddings = Settings.embed_model.get_text_embedding_batch(texts, show_progress=False)
        embedding_latency = time.perf_counter() - embedding_start

        # Vector DB insertion latency measures only the Chroma write/upsert part of the batch.
        insertion_start = time.perf_counter()
        collection.upsert(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
        insertion_latency = time.perf_counter() - insertion_start

        ingestion_observer.record_vector_batch(
            batch_index=batch_index,
            batch_size=len(batch_nodes),
            remaining_items=max(len(text_nodes) - start - len(batch_nodes), 0),
            embedding_latency_seconds=embedding_latency,
            insertion_latency_seconds=insertion_latency,
        )

    # Reopen the persisted vector store as a LlamaIndex index for retrieval tests after ingestion.
    index = VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=Settings.embed_model)
    with ingestion_observer.span(
        "ingestion.index_build_complete",
        {
            "index.state": "complete",
            "chunks.count": len(text_nodes),
            "batches.count": len(batch_starts),
            "collection.vector_count": collection.count(),
        },
    ):
        pass
    return index


# Group all ingestion spans under one parent span so Phoenix shows one run trace with child spans.
with ingestion_observer.span("ingestion.run", INGESTION_TRACE_ATTRIBUTES):
    index = build_or_update_vector_index(nodes, show_progress=True)
    collection = get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)
    dataset_upload_result = ingestion_observer.export_document_metrics_dataset()
    ingestion_summary = ingestion_observer.summary()

print(f"Collection name: {COLLECTION_NAME}")
print(f"Stored vector count: {collection.count()}")
print(f"Collection metadata: {vector_admin.get_collection_metadata(COLLECTION_NAME)}")
print(f"Ingestion summary: {ingestion_summary}")
print(f"Phoenix chunk metrics dataset action: {ingestion_observer.last_dataset_export_action}")
if ingestion_observer.last_dataset_export_error:
    print(f"Phoenix dataset export error: {ingestion_observer.last_dataset_export_error}")
print("Phoenix chunk metrics dataset upload:", "ok" if dataset_upload_result else "not available")

Deleted existing collection: news_chat_multilingual_e5_base


Embedding and writing Chroma batches:   0%|          | 0/43 [00:00<?, ?batch/s]

Collection name: news_chat_multilingual_e5_base
Stored vector count: 10972
Collection metadata: {'source_folder_count': 3, 'chunk_size': 800, 'chunk_overlap': 120, 'embedding_model': 'intfloat/multilingual-e5-base', 'source_folders': 'C:\\Program Files\\Studying\\coding\\RAG_project\\data\\hk01_news | C:\\Program Files\\Studying\\coding\\RAG_project\\data\\hk_free_press_news | C:\\Program Files\\Studying\\coding\\RAG_project\\data\\the_standard_news'}
Ingestion summary: {'documents': 5252, 'chunks': 10972, 'invalid_metadata_documents': 0, 'total_tokens_ingested_estimate': 1860434, 'vector_batches': 43, 'vectors_written': 10972, 'average_vectors_per_second': 3.5852264743922992, 'ram_mb_current': 2147.78515625}
Phoenix chunk metrics dataset action: failed
Phoenix dataset export error: create failed: timed out; append failed: timed out
Phoenix chunk metrics dataset upload: not available


## 4. Insert Chunks into ChromaDB

This cell creates a persistent ChromaDB collection and inserts the chunks through LlamaIndex. `RESET_COLLECTION = True` rebuilds this notebook's collection cleanly, which prevents duplicate chunks when you rerun the notebook. Set it to `False` only when you intentionally want to append new chunks.

In [7]:
RESET_COLLECTION = True


def get_chroma_collection(
    chroma_dir: Path,
    collection_name: str,
    reset_collection: bool = False,
    metadata: dict | None = None,
):
    """Create or load a persistent ChromaDB collection."""
    # PersistentClient writes vectors and metadata to disk instead of keeping them in memory.
    chroma_dir.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(chroma_dir))

    # Some ChromaDB versions raise NotFoundError when the collection is missing,
    # while older versions may raise ValueError. Catch both so reset stays safe.
    if reset_collection:
        try:
            client.delete_collection(collection_name)
            print(f"Deleted existing collection: {collection_name}")
        except (NotFoundError, ValueError):
            print(f"Collection did not exist yet: {collection_name}")

    # Chroma collection metadata must be scalar values (string/number/bool), not arrays.
    return client.get_or_create_collection(collection_name, metadata=metadata)


def sanitize_chroma_metadata(metadata: dict) -> dict:
    """Keep only Chroma-compatible scalar metadata values."""
    # Chroma rejects lists, dicts, and None values, so convert/drop them before insertion.
    scalar_metadata = {}
    for key, value in metadata.items():
        if isinstance(value, (str, int, float, bool)):
            scalar_metadata[key] = value
        elif value is not None:
            scalar_metadata[key] = str(value)
    return scalar_metadata


def build_or_update_vector_index(
    text_nodes: list[TextNode],
    chroma_dir: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
    reset_collection: bool = RESET_COLLECTION,
    batch_size: int = 256,
    show_progress: bool = True,
) -> VectorStoreIndex:
    """Embed nodes, write them to ChromaDB, and emit Phoenix ingestion metrics."""
    source_folders = sorted(str(news_dir) for news_dir in NEWS_DIRS)
    collection_metadata = {
        "embedding_model": EMBED_MODEL_NAME,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "source_folder_count": len(source_folders),
        # Serialize folders into one string because list metadata crashes Chroma.
        "source_folders": " | ".join(source_folders),
    }

    # ChromaVectorStore adapts the Chroma collection to LlamaIndex's vector store interface.
    collection = get_chroma_collection(chroma_dir, collection_name, reset_collection, metadata=collection_metadata)
    vector_store = ChromaVectorStore(chroma_collection=collection)

    if not text_nodes:
        return VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=Settings.embed_model)

    batch_starts = list(range(0, len(text_nodes), batch_size))
    batch_iterator = batch_starts
    if show_progress:
        batch_iterator = tqdm(batch_starts, desc="Embedding and writing Chroma batches", unit="batch")

    for batch_index, start in enumerate(batch_iterator, start=1):
        batch_nodes = text_nodes[start : start + batch_size]
        texts = [node.get_content() for node in batch_nodes]
        ids = [node.node_id for node in batch_nodes]
        metadatas = [sanitize_chroma_metadata(node.metadata or {}) for node in batch_nodes]

        # Embedding latency is traced separately from vector DB latency so Phoenix can show the bottleneck.
        embedding_start = time.perf_counter()
        embeddings = Settings.embed_model.get_text_embedding_batch(texts, show_progress=False)
        embedding_latency = time.perf_counter() - embedding_start

        # Vector DB insertion latency measures only the Chroma write/upsert part of the batch.
        insertion_start = time.perf_counter()
        collection.upsert(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
        insertion_latency = time.perf_counter() - insertion_start

        ingestion_observer.record_vector_batch(
            batch_index=batch_index,
            batch_size=len(batch_nodes),
            remaining_items=max(len(text_nodes) - start - len(batch_nodes), 0),
            embedding_latency_seconds=embedding_latency,
            insertion_latency_seconds=insertion_latency,
        )

    # Reopen the persisted vector store as a LlamaIndex index for retrieval tests after ingestion.
    index = VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=Settings.embed_model)
    with ingestion_observer.span(
        "ingestion.index_build_complete",
        {
            "index.state": "complete",
            "chunks.count": len(text_nodes),
            "batches.count": len(batch_starts),
            "collection.vector_count": collection.count(),
        },
    ):
        pass
    return index


# Group all ingestion spans under one parent span so Phoenix shows one run trace with child spans.
with ingestion_observer.span("ingestion.run", INGESTION_TRACE_ATTRIBUTES):
    index = build_or_update_vector_index(nodes, show_progress=True)
    collection = get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)
    dataset_upload_result = ingestion_observer.export_document_metrics_dataset()
    ingestion_summary = ingestion_observer.summary()

print(f"Collection name: {COLLECTION_NAME}")
print(f"Stored vector count: {collection.count()}")
print(f"Collection metadata: {vector_admin.get_collection_metadata(COLLECTION_NAME)}")
print(f"Ingestion summary: {ingestion_summary}")
print(f"Phoenix chunk metrics dataset action: {ingestion_observer.last_dataset_export_action}")
if ingestion_observer.last_dataset_export_error:
    print(f"Phoenix dataset export error: {ingestion_observer.last_dataset_export_error}")
print("Phoenix chunk metrics dataset upload:", "ok" if dataset_upload_result else "not available")

Deleted existing collection: news_chat_multilingual_e5_base


Embedding and writing Chroma batches:   0%|          | 0/43 [00:00<?, ?batch/s]

Collection name: news_chat_multilingual_e5_base
Stored vector count: 10972
Collection metadata: {'chunk_overlap': 120, 'embedding_model': 'intfloat/multilingual-e5-base', 'chunk_size': 800, 'source_folder_count': 3, 'source_folders': 'C:\\Program Files\\Studying\\coding\\RAG_project\\data\\hk01_news | C:\\Program Files\\Studying\\coding\\RAG_project\\data\\hk_free_press_news | C:\\Program Files\\Studying\\coding\\RAG_project\\data\\the_standard_news'}
Ingestion summary: {'documents': 5252, 'chunks': 10972, 'invalid_metadata_documents': 0, 'total_tokens_ingested_estimate': 1860434, 'vector_batches': 86, 'vectors_written': 21944, 'average_vectors_per_second': 3.638642986981544, 'ram_mb_current': 2030.796875}
Phoenix chunk metrics dataset action: created
Phoenix chunk metrics dataset upload: ok
